In [ ]:
import os
import torch

# 1. Clone the repository if not present
if not os.path.exists('main.py'):
    !git clone https://github.com/yangzeha/NLGCL.git
    %cd NLGCL

# 2. Setup Dependencies
# Note: We exclude torch_scatter/sparse from requirements.txt to install them separately with specific wheels
requirements_content = """colorlog
tensorboard
six
colorama
pyyaml
pandas
hyperopt==0.2.5
torch>=1.13.1
numpy<2.0.0
torch_geometric
# python>=3.9.18
"""
with open('requirements.txt', 'w') as f:
    f.write(requirements_content)

print("Installing base requirements...")
!pip install -r requirements.txt

print(f"Detected PyTorch Version: {torch.__version__}")
# Install torch-scatter/sparse using wheels to avoid long build times
# We try to construct a compatible wheel URL based on installed torch version
# Fallback to general link if specific one fails, or let pip build from source if no wheel found.
try:
    torch_version_suffix = torch.__version__.split('+')[0]
    # Assuming CUDA 12.1 as a common baseline for newer torch on Kaggle/Colab, or verify visually
    # Users can adjust the cu121 part if their environment differs significantly
    wheel_url = f"https://data.pyg.org/whl/torch-{torch_version_suffix}+cu121.html"
    print(f"Attempting to install scatter/sparse from {wheel_url}")
    !pip install torch-scatter torch-sparse -f {wheel_url}
except Exception as e:
    print("Wheel installation might have issues, falling back to standard install...")
    !pip install torch-scatter torch-sparse

# 3. Dataset Preprocessing (Without modifying source repo structure permanently)
# Create the directory structure RecBole expects
if not os.path.exists('dataset/QB-video'):
    os.makedirs('dataset/QB-video')

# Move the CSV file to the expected location and rename to .inter
if os.path.exists('dataset/QB-video.csv') and not os.path.exists('dataset/QB-video/QB-video.inter'):
    os.rename('dataset/QB-video.csv', 'dataset/QB-video/QB-video.inter')

# 4. Configuration Adjustment
# Modifying the configuration file to include field_separator
config_path = 'properties/QB-video.yaml'
if os.path.exists(config_path):
    with open(config_path, 'r') as f:
        content = f.read()
    
    # Check if field_separator is already there to avoid duplication
    if 'field_separator' not in content:
        # Inject field_separator after load_col block, using regex or string replacement
        # We look for the load_col block and add the separator
        new_content = content.replace('inter: [user_id, item_id]', 'inter: [user_id, item_id]\n\nfield_separator: ","')
        with open(config_path, 'w') as f:
            f.write(new_content)

# 5. Run Training
!python main.py --dataset QB-video